In [17]:
from dotenv import load_dotenv
import requests
import os

In [20]:
load_dotenv()

EXCHANGE_RATE_API_KEY = os.getenv("EXCHANGE_RATE_API_KEY")

if not EXCHANGE_RATE_API_KEY:
    raise ValueError("EXCHANGE_RATE_API_KEY is not set in the environment variables.")

EXCHANGE_RATE_BASE_URL = f"https://v6.exchangerate-api.com/v6/{EXCHANGE_RATE_API_KEY}/pair/"

def convert_currency(amount: float, currency_from: str, currency_to: str) -> str:
    response = requests.get(f"{EXCHANGE_RATE_BASE_URL}{currency_from.upper()}/{currency_to.upper()}/{amount}")
    status_code = response.status_code
    data = response.json()
    if data.get("result") == "error":
        error_type = data.get("error-type", "unknown")
        if error_type == "unsupported-code":
            raise requests.HTTPError(f"({status_code}): Unknown currency code: {currency_from.upper()} or {currency_to.upper()}.")
        elif error_type == "malformed-request":
            raise requests.HTTPError(f"({status_code}): Malformed request")
        elif error_type == "invalid-key":
            raise requests.HTTPError(f"({status_code}): Invalid API key")
        elif error_type == "inactive-account":
            raise requests.HTTPError(f"({status_code}): Account inactive")
        elif error_type == "quota-reached":
            raise requests.HTTPError(f"({status_code}): API quota reached. Please try again later.")
        else:
            raise requests.HTTPError(f"({status_code}): API error: {error_type}")

    conversion_result = data.get("conversion_result")
    conversion_rate = data.get("conversion_rate")

    return f"{amount} of {currency_from.upper()} converts to {conversion_result:.2f} {currency_to.upper()} using conversion rate {conversion_rate}"


In [13]:
print(convert_currency(100, "USD", "EUR"))

100 of USD converts to 86.24 EUR using conversion rate 0.8624
